In [ ]:
# Stock Price Prediction using LSTM

!pip install yfinance
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model


# ---------------------------------------------------------
# LOAD DATASET
# ---------------------------------------------------------
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download NVIDIA stock data directly from Yahoo Finance
ticker = "NVDA"

data = yf.download(
    ticker,
    period="10y",
    interval="1d"
)

print(data)


# ---------------------------------------------------------
# LOAD AND CLEAN DATASET
# ---------------------------------------------------------

def load_dataset():

    df = pd.read_csv('stock_data.csv')

    # Convert Date column to datetime
    df['Date'] = pd.to_datetime(df['Date'])

    # Sort according to date
    df = df.sort_values('Date')

    # Remove missing values
    df = df.dropna()

    # Select required features
    x_data = df[['Open', 'High', 'Low', 'Close', 'Volume']]

    return df, x_data


df, x_data = load_dataset()


print('Stock Data')
print(x_data.head())

print('\nDataset Shape:')
print(x_data.shape)


# ---------------------------------------------------------
# VISUALIZE STOCK PRICE
# ---------------------------------------------------------

plt.figure(figsize=(12, 5))

plt.plot(df['Date'], df['Close'])

plt.title('Stock Closing Price')
plt.xlabel('Date')
plt.ylabel('Closing Price')

plt.show()


# ---------------------------------------------------------
# SCALE DATA
# ---------------------------------------------------------

scaler = MinMaxScaler(feature_range=(0, 1))

scaled_data = scaler.fit_transform(x_data)


print('\nScaled Data')
print(scaled_data[:5])


# ---------------------------------------------------------
# CREATE TIME-SERIES SEQUENCES
# ---------------------------------------------------------

TIME_STEPS = 60


def create_sequences(data, time_steps):

    x = []
    y = []

    for i in range(time_steps, len(data)):

        # Previous 60 days
        x.append(data[i-time_steps:i])

        # Next day's closing price
        y.append(data[i, 3])

    return np.array(x), np.array(y)


x_data, y_data = create_sequences(
    scaled_data,
    TIME_STEPS
)


print('\nX Data Shape:')
print(x_data.shape)

print('\nY Data Shape:')
print(y_data.shape)


# ---------------------------------------------------------
# SPLIT DATASET
# ---------------------------------------------------------

train_size = int(len(x_data) * 0.8)

x_train = x_data[:train_size]
x_test = x_data[train_size:]

y_train = y_data[:train_size]
y_test = y_data[train_size:]


print('\nTrain Set')
print('X Train:', x_train.shape)
print('Y Train:', y_train.shape)

print('\nTest Set')
print('X Test:', x_test.shape)
print('Y Test:', y_test.shape)


# ---------------------------------------------------------
# ARCHITECTURE
# ---------------------------------------------------------

LSTM_OUT = 64

model = Sequential()

model.add(
    LSTM(
        LSTM_OUT,
        return_sequences=True,
        input_shape=(x_train.shape[1], x_train.shape[2])
    )
)

model.add(Dropout(0.2))

model.add(
    LSTM(32)
)

model.add(Dropout(0.2))

model.add(
    Dense(1)
)


model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mae']
)


print(model.summary())


# ---------------------------------------------------------
# SAVE BEST MODEL
# ---------------------------------------------------------

checkpoint = ModelCheckpoint(
    'LSTM_stock.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)


# ---------------------------------------------------------
# TRAIN MODEL
# ---------------------------------------------------------

history = model.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=10,
    validation_split=0.1,
    callbacks=[checkpoint]
)


# ---------------------------------------------------------
# LOAD BEST MODEL
# ---------------------------------------------------------

loaded_model = load_model('LSTM_stock.h5')


# ---------------------------------------------------------
# PREDICTION
# ---------------------------------------------------------

y_pred = loaded_model.predict(x_test)


# ---------------------------------------------------------
# CONVERT PREDICTIONS BACK TO ORIGINAL PRICE
# ---------------------------------------------------------

# Create dummy arrays because the scaler was trained
# using 5 features.

dummy_pred = np.zeros(
    (len(y_pred), 5)
)

dummy_actual = np.zeros(
    (len(y_test), 5)
)

# Put predicted/actual Close price in column 3
dummy_pred[:, 3] = y_pred[:, 0]
dummy_actual[:, 3] = y_test


# Inverse transform
predicted_prices = scaler.inverse_transform(
    dummy_pred
)[:, 3]

actual_prices = scaler.inverse_transform(
    dummy_actual
)[:, 3]


# ---------------------------------------------------------
# EVALUATION
# ---------------------------------------------------------

mse = mean_squared_error(
    actual_prices,
    predicted_prices
)

rmse = np.sqrt(mse)

mae = mean_absolute_error(
    actual_prices,
    predicted_prices
)


print('\nModel Evaluation')

print('Mean Squared Error:', mse)
print('Root Mean Squared Error:', rmse)
print('Mean Absolute Error:', mae)


# ---------------------------------------------------------
# PLOT ACTUAL VS PREDICTED
# ---------------------------------------------------------

plt.figure(figsize=(12, 5))

plt.plot(
    actual_prices,
    label='Actual Price'
)

plt.plot(
    predicted_prices,
    label='Predicted Price'
)

plt.title('Actual vs Predicted Stock Price')

plt.xlabel('Days')

plt.ylabel('Stock Price')

plt.legend()

plt.show()


# ---------------------------------------------------------
# PREDICT NEXT DAY STOCK PRICE
# ---------------------------------------------------------

last_60_days = scaled_data[-TIME_STEPS:]

last_60_days = np.reshape(
    last_60_days,
    (1, TIME_STEPS, 5)
)


next_day_prediction = loaded_model.predict(
    last_60_days
)


# Convert predicted value back to original price

dummy = np.zeros((1, 5))

dummy[0, 3] = next_day_prediction[0][0]

next_day_price = scaler.inverse_transform(
    dummy
)[0, 3]


print('\nPredicted Next Day Closing Price:')
print(next_day_price)

/tmp/ipykernel_1294/1209839719.py:28: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
ERROR:yfinance:Failed to get ticker 'NVDA' reason: Impersonating chrome150 is not supported
[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['NVDA']: ImpersonateError('Impersonating chrome150 is not supported')


Empty DataFrame
Columns: [(Adj Close, NVDA), (Close, NVDA), (High, NVDA), (Low, NVDA), (Open, NVDA), (Volume, NVDA)]
Index: []


FileNotFoundError: [Errno 2] No such file or directory: 'stock_data.csv'